### Imports

In [1]:

import torch
from torch import nn

from src.datasets.spike_dataset import SpikeDataset
from src.engine import benchmark_snn, train_one_epoch_snn, validate_snn
from src.models.snn_1d_encoded_classifier import SNN2DEncodedClassifier
from src.utils import get_split_dataloaders

### Constants

In [2]:
INPUT_DIR = '../../processed/audioMNIST'
MODEL_PATH = '../../models/best_snn.pth'

# Hyperparameters
LR = 0.001
NUM_EPOCHS = 20
SLOPE = 25

### Setting up device to use

In [3]:
device = torch.device(
    'cuda' if torch.cuda.is_available() else
    'mps' if torch.backends.mps.is_available() else
    'cpu'
)
print(f'Using device: {device}')

Using device: mps


### Training script

In [5]:
if __name__ == '__main__':
    dataset = SpikeDataset(data_dir=INPUT_DIR)
    train_dataloader, val_dataloader, test_dataloader = get_split_dataloaders(dataset)

    # Should have an extra dimension vs. the CNN for time at index 1
    features, labels = next(iter(train_dataloader))
    print(f'Features shape: {features.shape}')
    print(f'Labels shape: {labels.shape}')
    print()

    model = SNN2DEncodedClassifier(slope=SLOPE).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    criterion = nn.CrossEntropyLoss()

    print("Training SNN...")
    best_accuracy = 0.0
    for epoch in range(NUM_EPOCHS):
        saved = False

        print(f'[Epoch {epoch + 1}/{NUM_EPOCHS}]')
        train_loss, train_accuracy = train_one_epoch_snn(device, model, criterion, optimizer, train_dataloader)
        val_loss, val_accuracy = validate_snn(device, model, criterion, val_dataloader)

        if val_accuracy > best_accuracy:
            saved = True
            best_accuracy = val_accuracy
            torch.save(model.state_dict(), MODEL_PATH)

        print(f'Train Loss: {train_loss:.2f} | Train Accuracy: {train_accuracy:.2f}% | Val Loss: {val_loss:.2f} | Val Accuracy: {val_accuracy:.2f}%')
        print()
    print(f'Best model had an accuracy of {best_accuracy:.2f}%.')
    print(f'Running final test...')

    checkpoint = torch.load(MODEL_PATH, map_location=device, weights_only=True)
    model.load_state_dict(checkpoint, strict=True)
    model.to(device)

    test_accuracy, avg_acs_per_inference = benchmark_snn(device, model, test_dataloader)

    print(f'Test accuracy: {test_accuracy:.2f}% | Total ACs: {avg_acs_per_inference:.0f}')

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Features shape: torch.Size([64, 2, 27, 64])
Labels shape: torch.Size([64])

Training SNN...
[Epoch 1/20]


Validating:   0%|          | 0/47 [00:00<?, ?batches/s]/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
Validating: 100%|██████████| 47/47 [00:04<00:00,  9.86batches/s]


Train Loss: 2.30 | Train Accuracy: 14.11% | Val Loss: 2.01 | Val Accuracy: 28.63%

[Epoch 2/20]


Validating: 100%|██████████| 47/47 [00:01<00:00, 25.12batches/s]


...

Train Loss: 0.63 | Train Accuracy: 77.69% | Val Loss: 0.56 | Val Accuracy: 80.10%

[Epoch 6/20]


Validating: 100%|██████████| 47/47 [00:01<00:00, 24.83batches/s]


Train Loss: 0.56 | Train Accuracy: 80.15% | Val Loss: 0.52 | Val Accuracy: 80.17%

[Epoch 7/20]


Training:  65%|██████▌   | 244/375 [00:22<00:12, 10.83batches/s]


KeyboardInterrupt: 